# 4.Database Integration

## Question 4: Database Integration


## Introduction


This section demonstrates database integration for the South African GDP growth and employment dataset. A SQLite database is used to store the cleaned dataset in a structured, queryable format.

The process covers:
- Creating a database table with an appropriate schema
- Loading the cleaned CSV data into the database, with error handling
- Querying the database using several different SQL techniques (basic selection, aggregation, subqueries, and filtering)
- Safely updating and deleting records using parameterised queries
- Loading the database contents back into Pandas for further analysis
- Exporting the final data back out to CSV

This shows that the dataset can move reliably between file storage, a relational database, and Python for analysis — rather than existing only as a static spreadsheet.

#### 4.1 Connecting 

In [1]:
import sqlite3
import csv

In [2]:
conn = sqlite3.connect("GDP_Employment_South_Africa.db")
cursor = conn.cursor()

#### 4.2 Creating a table

In [3]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS gdp_employment (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    country TEXT,
    year INTEGER,
    gdp_growth REAL,
    employment_thousands REAL,
    sector TEXT
)
''')
conn.commit()

####  4.3 Loading the CSV into the table (with error handling)

In [4]:
try:
    with open('../../data/final_clean/GDP_Employment_South_Africa_Cleaned.csv', 'r') as file:
        csv_reader = csv.reader(file)
        next(csv_reader)  # skip header row
        for row in csv_reader:
            cursor.execute('''
                INSERT INTO gdp_employment (country, year, gdp_growth, employment_thousands, sector)
                VALUES (?, ?, ?, ?, ?)
            ''', row)
    conn.commit()
    print("Data loaded successfully.")
except FileNotFoundError:
    print("Error: CSV file not found. Check the path.")
except sqlite3.Error as e:
    print(f"Database error: {e}")

Data loaded successfully.


![Database schema](images/schema_screenshot.png)

#### 4.4 Query the database (different query types)

##### Basic select

In [5]:
cursor.execute("SELECT * FROM gdp_employment LIMIT 5")
for row in cursor.fetchall():
    print(row)

(1, 'South Africa', 2017, 1.157946951817351, 16536.302734375, 'Total')
(2, 'South Africa', 2017, 1.157946951817351, 849.6207731755255, 'Agriculture')
(3, 'South Africa', 2017, 1.157946951817351, 414.533397544058, 'Mining')
(4, 'South Africa', 2017, 1.157946951817351, 1855.8184205482744, 'Manufacturing')
(5, 'South Africa', 2017, 1.157946951817351, 187.1115981720818, 'Utilities')


![Database schema](images/basic.png)

##### Aggregate query (Average employment per sector)

In [6]:
cursor.execute('''
    SELECT sector, ROUND(AVG(employment_thousands), 2) AS avg_employment
    FROM gdp_employment
    GROUP BY sector
    ORDER BY avg_employment DESC
''')
for row in cursor.fetchall():
    print(row)

('Total', 9352.85)
('Sector: Total', 9352.85)
('Sector: Other services', 2232.17)
('Other services', 2232.17)
('Trade services', 2071.34)
('Sector: Trade services', 2071.34)
('Sector: Manufacturing', 1473.76)
('Manufacturing', 1473.76)
('Sector: Agriculture', 1039.69)
('Agriculture', 1039.69)
('Sector: Finance and business services', 831.6)
('Finance and business services', 831.6)
('Sector: Construction', 642.53)
('Construction', 642.53)
('Transport services', 515.61)
('Sector: Transport services', 515.61)
('Sector: Mining', 473.72)
('Mining', 473.72)
('Utilities', 76.44)
('Sector: Utilities', 72.42)


![Database schema](images/avg_employment.png)

In [7]:
cursor.execute('''
    SELECT year, gdp_growth
    FROM gdp_employment
    WHERE sector = 'Total'
    AND gdp_growth > (
        SELECT AVG(gdp_growth) FROM gdp_employment WHERE sector = 'Total'
    )
    ORDER BY year
''')
for row in cursor.fetchall():
    print(row)

(1961, 3.8447341417408873)
(1961, 3.8447341417408873)
(1962, 6.177930850427458)
(1962, 6.177930850427458)
(1963, 7.373709248772229)
(1963, 7.373709248772229)
(1964, 7.939608644806768)
(1964, 7.939608644806768)
(1965, 6.122798096014421)
(1965, 6.122798096014421)
(1966, 4.438386089468892)
(1966, 4.438386089468892)
(1967, 7.196523065141065)
(1967, 7.196523065141065)
(1968, 4.153372982933462)
(1968, 4.153372982933462)
(1969, 4.715902867222411)
(1969, 4.715902867222411)
(1970, 5.248661431182924)
(1970, 5.248661431182924)
(1971, 4.278934402969824)
(1971, 4.278934402969824)
(1973, 4.571944746372708)
(1973, 4.571944746372708)
(1974, 6.111122103721883)
(1974, 6.111122103721883)
(1979, 3.7905192640578207)
(1979, 3.7905192640578207)
(1980, 6.62058342724157)
(1980, 6.62058342724157)
(1981, 5.360791059640448)
(1981, 5.360791059640448)
(1984, 5.099151630014774)
(1984, 5.099151630014774)
(1988, 4.2001096483818685)
(1988, 4.2001096483818685)
(1994, 3.20000000297172)
(1994, 3.20000000297172)
(1995, 3.1

![Database schema](images/sub_query.png)

##### Filtered query - GDP growth trend for a specific sector

In [8]:
cursor.execute('''
    SELECT year, gdp_growth
    FROM gdp_employment
    WHERE sector = 'Sector: Total'
    ORDER BY year
''')
rows = cursor.fetchall()
print(rows[:5])

[(1961, 3.8447341417408873), (1962, 6.177930850427458), (1963, 7.373709248772229), (1964, 7.939608644806768), (1965, 6.122798096014421)]


![Database schema](images/specific_sector.png)

#### 4.5 Updating records safely (parameterized, not string-formatted)

In [9]:
sectors = [row[0] for row in cursor.execute("SELECT DISTINCT sector FROM gdp_employment")]

for s in sectors:
    clean = s.replace("Sector: ", "")
    cursor.execute(
        "UPDATE gdp_employment SET sector = ? WHERE sector = ?",
        (clean, s)
    )

conn.commit()
print("Sector labels cleaned.")

Sector labels cleaned.


#### 4.6 Deleting records safely

In [10]:
cursor.execute("SELECT COUNT(*) FROM gdp_employment WHERE employment_thousands IS NULL OR employment_thousands = 0")
print("Rows to delete:", cursor.fetchone()[0])

cursor.execute("DELETE FROM gdp_employment WHERE employment_thousands IS NULL")
conn.commit()
print(f"{cursor.rowcount} rows deleted.")

Rows to delete: 0
0 rows deleted.


In [11]:
cursor.execute("DELETE FROM gdp_employment WHERE sector = 'Utilities' AND year < 1965")
conn.commit()
print(f"{cursor.rowcount} rows deleted.")

4 rows deleted.


#### 4.7 Loading the database back into Pandas

In [12]:
import pandas as pd

df = pd.read_sql_query("SELECT * FROM gdp_employment", conn)
print(df.shape)
df.head()

(1698, 6)


,id,country,year,gdp_growth,employment_thousands,sector
0,1,South Africa,2017,1.157947,16536.302734,Total
1,2,South Africa,2017,1.157947,849.620773,Agriculture
2,3,South Africa,2017,1.157947,414.533398,Mining
3,4,South Africa,2017,1.157947,1855.818421,Manufacturing
4,5,South Africa,2017,1.157947,187.111598,Utilities


#### 4.8 Exporting & closing

In [13]:
df.to_csv("gdp_employment_from_db.csv", index=False)
conn.close()
print("Exported and connection closed.")

Exported and connection closed.


## Conclusion

This section successfully built a working SQLite database from the cleaned South African GDP and employment dataset. A table was created with an appropriate schema, and all rows from the cleaned CSV file were loaded in successfully, with error handling in place in case the source file could not be found.

Several types of SQL queries were demonstrated on the database: a basic selection, an aggregate query calculating average employment per sector, a subquery identifying years with above-average GDP growth, and a filtered query tracking GDP growth over time for a specific sector. Records were also updated safely — cleaning the sector labels using parameterised queries rather than raw string formatting — and outdated records (Utilities sector data before 1965) were safely deleted, removing 4 rows.

Finally, the cleaned database contents were loaded back into a Pandas DataFrame and exported to CSV, confirming that data can move reliably between the database and the rest of the analysis pipeline. This fulfils the assignment's requirement to build and query a database, update and delete records safely, and load database data into Pandas.